# Robustez adversarial en la detección de cáncer de próstata

Clasificación de significancia histológica en imágenes de próstata con **ResNet50** (transfer learning), puesta a prueba con un ataque adversario **FGSM** y defendida con dos estrategias comparadas: entrenamiento con datos mixtos (original + atacado) y fine-tuning.

Caso de estudio completo, con las decisiones de diseño explicadas:
https://fuzzyfrog.ai/es/ai-lab/proyectos/salud/deteccion-cancer-prostata-robustez-adversarial/

Dataset de referencia (Kaggle):
https://www.kaggle.com/datasets/tgprostata/transverse-plane-prostate-dataset

**Estructura de este notebook:**
1. Setup y carga de datos
2. Modelo base (sin defensa)
3. Ataque adversario FGSM
4. Defensa adversarial (datos mixtos + fine-tuning)
5. Evaluación de robustez (accuracy, AUC-ROC, métricas ART)
6. Demo rápido: comparar predicción con/sin defensa sobre una imagen atacada

## 1. Setup y carga de datos

Estructura de carpetas esperada (ajusta `ROOT_DIR` a donde tengas el dataset, por ejemplo en Google Drive si corres esto en Colab):
```
ROOT_DIR/
  train/
    significant/
    notsignificant/
  validation/
    significant/
    notsignificant/
```

In [ ]:
# Si corres esto en Google Colab, monta tu Drive:
# from google.colab import drive
# drive.mount('/content/gdrive')

!pip install adversarial-robustness-toolbox[tensorflow] -q

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy
from sklearn.metrics import roc_curve, roc_auc_score

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
tf.random.set_seed(SEED)

ROOT_DIR = '/content/gdrive/MyDrive/ML/Dataset/Prostate Dataset'  # <-- ajusta esta ruta
train_dir = ROOT_DIR + '/train'
val_dir = ROOT_DIR + '/validation'

In [ ]:
# Generadores de datos: rescale simple (baseline)
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', color_mode='rgb', seed=SEED,
)
valid_data = val_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', color_mode='rgb', seed=SEED, shuffle=False,
)

print('Clases:', train_data.class_indices)
print('Muestras de entrenamiento:', train_data.samples)
print('Muestras de validación:', valid_data.samples)

**Nota de preprocesamiento:** también se probó una variante con conversión RGB→HSV y ecualización del canal de valor, buscando mejorar el contraste del tejido. No mostró una mejora consistente frente al rescale simple, así que se descartó como pipeline por default (se deja la función por si quieres probarla).

In [ ]:
def rgb_to_hsv_equalized(img: np.ndarray) -> np.ndarray:
    hsv_img = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2HSV)
    hsv_img[:, :, 2] = cv2.equalizeHist(hsv_img[:, :, 2])
    return cv2.cvtColor(hsv_img, cv2.COLOR_HSV2RGB)

## 2. Modelo base (sin defensa)

ResNet50 preentrenado en ImageNet + capas densas con dropout, para clasificación binaria. Este es el modelo que después se ataca con FGSM.

In [ ]:
def build_base_model(trainable_backbone: bool = False) -> tf.keras.Model:
    base_model = ResNet50(include_top=False, pooling='avg', weights='imagenet')
    base_model.trainable = trainable_backbone

    x = base_model.output
    x = Dense(2048, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

modelo_sin_defensa = build_base_model(trainable_backbone=False)
modelo_sin_defensa.summary()

In [ ]:
historial_sin_defensa = modelo_sin_defensa.fit(
    train_data,
    epochs=100,
    validation_data=valid_data,
    verbose=1,
)

In [ ]:
def visualize_training_history(history, title_prefix=''):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharex=True)
    ax1.plot(epochs_range, acc, label='Train accuracy')
    ax1.plot(epochs_range, val_acc, label='Val accuracy')
    ax1.set_title(f'{title_prefix} Accuracy'); ax1.legend()

    ax2.plot(epochs_range, loss, label='Train loss')
    ax2.plot(epochs_range, val_loss, label='Val loss')
    ax2.set_title(f'{title_prefix} Loss'); ax2.legend()
    plt.tight_layout(); plt.show()

visualize_training_history(historial_sin_defensa, title_prefix='ResNet50 sin defensa —')

In [ ]:
loss_limpio, acc_limpio = modelo_sin_defensa.evaluate(valid_data, verbose=0)
print(f'Accuracy en set limpio: {acc_limpio:.4f}')

modelo_sin_defensa.save('modelo_resnet50_sin_defensa.h5')
print('Modelo guardado.')

## 3. Ataque adversario FGSM

**FGSM (Fast Gradient Sign Method):** perturba la imagen en la dirección del gradiente de la función de pérdida respecto a la entrada, escalado por `epsilon`. El resultado es ruido casi imperceptible al ojo humano que puede cambiar la predicción.

**Criterio aplicado sobre epsilon:** se eligió balanceando imperceptibilidad visual y efectividad del ataque, verificando manualmente que la perturbación no fuera evidente a simple vista.

In [ ]:
DEFAULT_EPSILON = 0.05

def fgsm_attack(image: np.ndarray, true_label: float, model: tf.keras.Model,
                 epsilon: float = DEFAULT_EPSILON) -> tf.Tensor:
    image_tensor = tf.convert_to_tensor(image)
    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        prediction = model(image_tensor)
        label_tensor = tf.constant([[true_label]], dtype=tf.float32)
        loss = binary_crossentropy(label_tensor, prediction)
    gradient = tape.gradient(loss, image_tensor)
    perturbation = epsilon * tf.sign(gradient)
    perturbed_image = image_tensor + perturbation
    return tf.clip_by_value(perturbed_image, 0.0, 1.0)

def generate_adversarial_batch(images, labels, model, epsilon=DEFAULT_EPSILON):
    adversarial_images = np.zeros_like(images)
    for i in range(len(images)):
        img = np.expand_dims(images[i], axis=0)
        adv = fgsm_attack(img, labels[i], model, epsilon=epsilon)
        adversarial_images[i] = adv.numpy()[0]
    return adversarial_images

### Ejemplo visual: una imagen original vs. atacada

In [ ]:
valid_data.reset()
batch_images, batch_labels = next(valid_data)

idx = 0
original_image = np.expand_dims(batch_images[idx], axis=0)
true_label = batch_labels[idx]

perturbed_image = fgsm_attack(original_image, true_label, modelo_sin_defensa, epsilon=DEFAULT_EPSILON)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_image[0]); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(perturbed_image[0].numpy()); axes[1].set_title(f'Atacada (ε={DEFAULT_EPSILON})'); axes[1].axis('off')
plt.tight_layout(); plt.show()

pred_original = modelo_sin_defensa.predict(original_image, verbose=0)[0][0]
pred_atacada = modelo_sin_defensa.predict(np.expand_dims(perturbed_image[0], 0), verbose=0)[0][0]
print(f'Predicción original: {pred_original:.4f} | Predicción atacada: {pred_atacada:.4f}')

### Generar el set de validación atacado completo (para evaluación y defensa)

In [ ]:
x_val_adv = generate_adversarial_batch(batch_images, batch_labels, modelo_sin_defensa, epsilon=DEFAULT_EPSILON)
y_val_adv = batch_labels

loss_atacado, acc_atacado = modelo_sin_defensa.evaluate(x_val_adv, y_val_adv, verbose=0)
print(f'Accuracy en set limpio:  {acc_limpio:.4f}')
print(f'Accuracy en set atacado: {acc_atacado:.4f}')
print(f'Caída de accuracy: {(acc_limpio - acc_atacado) * 100:.1f} puntos porcentuales')

## 4. Defensa adversarial

Dos estrategias comparadas:
- **(A) Entrenamiento con datos mixtos** — combina imágenes originales y atacadas en el mismo set de entrenamiento, en vez de reemplazar el set original por completo, para no sacrificar desempeño en el caso limpio.
- **(B) Fine-tuning** — parte del backbone congelado y ajusta con menos épocas.

**Nota de iteración:** las primeras corridas con 100 épocas sobreajustaban al set combinado, perdiendo desempeño en el caso limpio. Se redujo a 50 épocas con `steps_per_epoch` reducido, lo que dio mejor balance.

In [ ]:
def build_defended_model() -> tf.keras.Model:
    model = Sequential([
        ResNet50(include_top=False, pooling='avg', weights='imagenet'),
        Flatten(),
        BatchNormalization(),
        Dense(2048, activation='relu'),
        BatchNormalization(),
        Dense(1024, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_fine_tuned_model(freeze_backbone: bool = True) -> tf.keras.Model:
    resnet_model = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', pooling='avg')
    resnet_model.trainable = not freeze_backbone
    model = Sequential([
        resnet_model,
        Flatten(),
        Dense(2048, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Generador combinado: alterna entre batches del train original y del set atacado
def generador_combinado(train_gen, x_adv, y_adv, batch_size=32):
    adv_idx = 0
    while True:
        x_batch, y_batch = next(train_gen)
        end = min(adv_idx + batch_size, len(x_adv))
        x_adv_batch = x_adv[adv_idx:end]
        y_adv_batch = y_adv[adv_idx:end]
        adv_idx = end if end < len(x_adv) else 0
        if len(x_adv_batch) > 0:
            x_batch = np.concatenate([x_batch, x_adv_batch], axis=0)
            y_batch = np.concatenate([y_batch, y_adv_batch], axis=0)
        yield x_batch, y_batch

train_gen_combinado = generador_combinado(train_data, x_val_adv, y_val_adv)

In [ ]:
# Estrategia A: entrenamiento con datos mixtos
modelo_con_defensa = build_defended_model()

historial_con_defensa = modelo_con_defensa.fit(
    train_gen_combinado,
    epochs=50,
    steps_per_epoch=20,
    validation_data=(x_val_adv, y_val_adv),
    verbose=1,
)

visualize_training_history(historial_con_defensa, title_prefix='ResNet50 con defensa —')

In [ ]:
# Estrategia B: fine-tuning
modelo_fine_tuned = build_fine_tuned_model(freeze_backbone=True)

historial_fine_tuned = modelo_fine_tuned.fit(
    train_gen_combinado,
    epochs=50,
    steps_per_epoch=20,
    validation_data=(x_val_adv, y_val_adv),
    verbose=1,
)

visualize_training_history(historial_fine_tuned, title_prefix='ResNet50 fine-tuned —')

In [ ]:
modelo_con_defensa.save('modelo_resnet50_con_defensa.h5')
modelo_fine_tuned.save('modelo_resnet50_fine_tuned.h5')
print('Modelos guardados.')

## 5. Evaluación de robustez

Compara los tres modelos con accuracy, AUC-ROC, y las métricas de robustez del Adversarial Robustness Toolbox (ART): `loss_sensitivity` y `empirical_robustness`. El accuracy no basta: estas métricas cuantifican qué tan fácil es engañar al modelo, no solo qué tan bien clasifica.

In [ ]:
def plot_roc_curve(y_true, y_pred_proba, title='Curva ROC'):
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel('Tasa de falsos positivos'); plt.ylabel('Tasa de verdaderos positivos')
    plt.title(title); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()
    return roc_auc

modelos = {
    'sin_defensa': modelo_sin_defensa,
    'con_defensa': modelo_con_defensa,
    'fine_tuning': modelo_fine_tuned,
}

resultados = {}
for nombre, modelo in modelos.items():
    loss, acc = modelo.evaluate(x_val_adv, y_val_adv, verbose=0)
    y_pred = modelo.predict(x_val_adv, verbose=0).ravel()
    auc = plot_roc_curve(y_val_adv, y_pred, title=f'ROC — {nombre} (set atacado)')
    resultados[nombre] = {'accuracy_atacado': acc, 'auc_atacado': auc}

print(resultados)

In [ ]:
from art.estimators.classification import KerasClassifier
from art.metrics import loss_sensitivity, empirical_robustness

for nombre, modelo in modelos.items():
    classifier = KerasClassifier(model=modelo, clip_values=(0, 1))
    sensitivity = loss_sensitivity(classifier, x_val_adv, y_val_adv)
    robustness = empirical_robustness(classifier, x_val_adv, attack_name='FastGradientMethod')
    resultados[nombre]['loss_sensitivity'] = float(sensitivity)
    resultados[nombre]['empirical_robustness'] = float(robustness)

tabla_final = pd.DataFrame(resultados).T
tabla_final.to_csv('metricas_robustez_comparacion.csv')
tabla_final

**Lectura del resultado:** un accuracy alto en el set atacado no es suficiente si `loss_sensitivity` sigue siendo alto — significa que el modelo sigue siendo fácil de mover con perturbaciones pequeñas, aunque en este batch particular no haya cambiado de clase. La estrategia de defensa recomendada es la que mejor balancea las cuatro columnas, no la que gana en una sola.

## 6. Demo rápido: comparar predicción con/sin defensa

Toma una imagen cualquiera del set de validación, la ataca con FGSM, y compara la predicción del modelo sin defensa contra la del modelo con defensa — en la imagen limpia y en la imagen atacada. Esto es lo mismo que haría un endpoint `/comparar` en un demo funcional (ver el caso de estudio para la versión FastAPI).

In [ ]:
def comparar_prediccion(idx: int, epsilon: float = DEFAULT_EPSILON):
    image = np.expand_dims(batch_images[idx], axis=0)
    label = batch_labels[idx]

    atacada = fgsm_attack(image, label, modelo_sin_defensa, epsilon=epsilon).numpy()

    def pred(modelo, img):
        p = float(modelo.predict(img, verbose=0)[0][0])
        clase = 'significant' if p >= 0.5 else 'notsignificant'
        return clase, round(p, 4)

    print('--- Imagen limpia ---')
    print('Sin defensa:', pred(modelo_sin_defensa, image))
    print('Con defensa:', pred(modelo_con_defensa, image))
    print('--- Imagen atacada (FGSM, epsilon=%.2f) ---' % epsilon)
    print('Sin defensa:', pred(modelo_sin_defensa, atacada))
    print('Con defensa:', pred(modelo_con_defensa, atacada))

comparar_prediccion(idx=0)